# 🎬 ZeusDL — BangBros Bulk Download + Telegram Upload

This notebook:
1. Installs ZeusDL and dependencies
2. Downloads all videos from a BangBros studio playlist
3. Uploads them to your Telegram channel

**Requirements:**
- A BangBros premium account (export cookies from your browser)
- A Telegram bot token (create one at @BotFather)
- Your Telegram channel/chat ID

> ℹ️ This Colab instance acts as a **Hermes agent** — you can also
> connect it to your Zeus Mastermind to receive orders remotely.


In [ ]:
# ── Cell 1: Install ZeusDL ────────────────────────────────────────────────
!pip install -q 'zeusdl[default]'
print('✅ ZeusDL installed')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os

# BangBros settings
STUDIO_URL  = 'https://site-ma.bangbros.com/scenes?addon=5971'  # Change addon ID
COOKIES_PATH = '/content/bangbros_cookies.txt'  # Upload your cookies file
OUTPUT_DIR   = '/content/downloads'
QUALITY      = '1080p'   # 360p | 480p | 720p | 1080p | best
WORKERS      = 3         # Parallel downloads

# Telegram settings
BOT_TOKEN  = 'YOUR_BOT_TOKEN_HERE'       # From @BotFather
CHANNEL_ID = '-1001234567890'            # Your channel/chat ID

# Optional: Hermes Mastermind connection
HERMES_AGENT_ID  = 'colab-1'                    # Unique name for this Colab
HERMES_MASTER_URL = ''                           # Leave empty to run standalone

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config OK — output: {OUTPUT_DIR}')

In [ ]:
# ── Cell 3: Upload your cookies file ─────────────────────────────────────
# Run this cell to upload your BangBros cookies.txt
from google.colab import files
uploaded = files.upload()
for fn in uploaded:
    os.rename(fn, COOKIES_PATH)
    print(f'✅ Cookies saved to {COOKIES_PATH}')

In [ ]:
# ── Cell 4: Download all videos ───────────────────────────────────────────
from zeusdl.bulk_download import BulkDownloader

dl = BulkDownloader(
    output_dir=OUTPUT_DIR,
    cookies=COOKIES_PATH,
    workers=WORKERS,
    quality=QUALITY,
    sleep=2.0,
)

result = dl.run(STUDIO_URL)
print(f"\n✅ Download summary: {result}")

In [ ]:
# ── Cell 5: Upload to Telegram ────────────────────────────────────────────
from zeusdl.telebot.uploader import TelegramUploader

uploader = TelegramUploader(
    bot_token=BOT_TOKEN,
    channel=CHANNEL_ID,
)

result = uploader.upload_directory(
    OUTPUT_DIR,
    caption=f'ZeusDL — {QUALITY} batch'
)
print(f"\n✅ Upload summary: {result}")

In [ ]:
# ── Cell 6 (Optional): Run as a Hermes agent ──────────────────────────────
# Connect this Colab to your Zeus Mastermind to receive orders remotely.
# Start the Mastermind first on your machine:
#   zeusdl hermes mastermind --port 8765

if HERMES_MASTER_URL:
    from zeusdl.hermes.agent import HermesAgent
    agent = HermesAgent(
        agent_id=HERMES_AGENT_ID,
        master_url=HERMES_MASTER_URL,
    )
    agent.set_var('quality', QUALITY)
    agent.set_var('cookies', COOKIES_PATH)
    agent.set_var('output', OUTPUT_DIR)
    agent.set_var('telegram_token', BOT_TOKEN)
    agent.set_var('telegram_channel', CHANNEL_ID)
    print(f'🚀 Starting Hermes agent "{HERMES_AGENT_ID}"…')
    agent.start()   # blocks until stopped
else:
    print('ℹ️ HERMES_MASTER_URL not set — running in standalone mode.')

In [ ]:
# ── Cell 7 (Optional): Run a .zeus script inline ──────────────────────────
from zeusdl.zscript.runner import ZeusScriptRunner

SCRIPT = """
set quality  720p
set workers  2
set output   /content/downloads_720p

download https://site-ma.bangbros.com/scenes?addon=5971
    cookies /content/bangbros_cookies.txt

push telegram
    token   YOUR_BOT_TOKEN_HERE
    channel -1001234567890
"""

runner = ZeusScriptRunner()
runner.run_string(SCRIPT)